## Acquisition

In [1]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import csv
from datetime import datetime
from dataclasses import dataclass


def create_new_sampling_file(path, name):
    # complete file name with unique ID
    file_id = datetime.now()
    file_name = (path + name + "_" + file_id.strftime("%y-%m-%d_%H-%M-%S") + ".csv")

    # opening file in writing mode
    file = open(file_name, 'w', newline="")

    # header fields
    fields = ['voltage1 (V)', 'voltage2 (V)']
    csv.DictWriter(file, fieldnames=fields).writeheader()

    # closing file
    file.close()

    return file_name


def acquire_window(ch1, ch2, window_size, writer):
    # EMG data for current window
    data1 = []
    data2 = []

    # acquisition loop
    for i in range(window_size):
        data1.append(ch1.voltage)
        data2.append(ch2.voltage)
        writer.writerow([data1[i], data2[i]])

    return data1, data2


def acquire_training_dataset(ch1, ch2, window_size, num_window, file_name):
    # file opening in append mode
    file = open(file_name, 'a', newline="")
    training_writer = csv.writer(file)

    # acquisition loop
    print("debut acquisition")
    start_time = time.time()

    for window in range(num_window):
        acquire_window(ch1, ch2, window_size, training_writer)

    end_time = time.time()
    file.close()

    # frequency sampling calculation
    total_samples = window_size * num_window
    elapsed_time = end_time - start_time
    fs = total_samples / elapsed_time
    print("frequence acquisition : " + str(round(fs, 2)) + " sps")

    return fs


def visualize_sampling(file):
    data = pd.read_csv(file)

    v1 = data["voltage1 (V)"]
    v2 = data["voltage2 (V)"]
    x = range(0, len(v1))

    plt.plot(x, v1, label="voltage1 (V)")
    plt.plot(x, v2, label="voltage2 (V)")

    plt.title("Plot of Sensors Voltages")
    plt.legend()
    plt.show()

## Signal processing

In [218]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import glob
import os
from collections import Counter
from sklearn.metrics import accuracy_score
#from scipy.signal import butter, filtfilt, iirnotch



from scipy.signal import butter, lfilter, iirnotch

def filter_emg(signal, fs):
    """
    Filtrage EMG complet (notch + passe-bande) - CAUSAL

    Parameters:
    - signal : array (signal brut)
    - fs : fréquence d'échantillonnage (Hz)

    Returns:
    - signal filtré
    """

    # ---------- 1. NOTCH FILTER (bruit secteur) ----------
    notch_freq=60
    Q = 30  # facteur de qualité
    # notch_freq=60
    # Q = 25  # facteur de qualité
    b_notch, a_notch = iirnotch(notch_freq, Q, fs)
    signal = lfilter(b_notch, a_notch, signal)

    # ---------- 2. PASSE-BANDE ----------
    lowcut=20
    highcut=450
    # lowcut=25
    # highcut=450
    order=4
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist

    b_band, a_band = butter(order, [low, high], btype='band')
    signal = lfilter(b_band, a_band, signal)

    return signal

def preprocess_signal(signal, fs=1000):
    """
    Preprocess one EMG channel

    Parameters:
    - signal: raw EMG signal (1D array)
    - fs: sampling frequency (Hz)

    Returns:
    - filtered_signal
    """

    signal = np.array(signal)

    # -----------------------------
    # 1. Band-pass filter (20–450 Hz)
    # -----------------------------
    lowcut = 20
    highcut = 450

    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(4, [low, high], btype='band')
    signal = filtfilt(b, a, signal)

    # -----------------------------
    # 2. Notch filter (60 Hz)
    # -----------------------------
    notch_freq = 60  # Canada → 60 Hz
    Q = 30.0         # Quality factor

    b_notch, a_notch = iirnotch(notch_freq / nyquist, Q)
    signal = filtfilt(b_notch, a_notch, signal)

    # -----------------------------
    # 3. Optional: normalization
    # -----------------------------
    signal = (signal - np.mean(signal)) / (np.std(signal) + 1e-8)

    return signal


def preprocess_signal_old(signal, cutoff_freq_normalized=0.5, order=4):
    return signal
    # Apply a low-pass Butterworth filter
    # cutoff_freq_normalized is between 0 and 1, where 1 is the Nyquist frequency (sampling_frequency / 2)
    b, a = butter(order, cutoff_freq_normalized, btype='low', analog=False)
    filtered_signal = filtfilt(b, a, signal)

    # Normalize the signal (z-score normalization)
    mean = np.mean(filtered_signal)
    std = np.std(filtered_signal)
    if std == 0: # Avoid division by zero if signal is constant
        normalized_signal = filtered_signal - mean
    else:
        normalized_signal = (filtered_signal - mean) / std
    return normalized_signal





# Train


In [252]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Import models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC




def extract_features(signal):
    """
    Extraction de features EMG pour une fenêtre
    """

    # Mean Absolute Value
    mav = np.mean(np.abs(signal))

    # Zero Crossing
    zc = np.sum(signal[:-1] * signal[1:] < 0)

    # Waveform Length
    wl = np.sum(np.abs(np.diff(signal)))

    # RMS
    rms = np.sqrt(np.mean(signal**2))

    # Slope Sign Changes
    ssc = np.sum(((signal[1:-1] - signal[:-2]) * (signal[1:-1] - signal[2:])) > 0)

    # Variance
    var = np.var(signal)

    # Willison Amplitude
    # threshold = 0.01
    threshold = 0.05
    wamp = np.sum(np.abs(signal[1:] - signal[:-1]) > threshold)

    # IEMG
    iemg = np.sum(np.abs(signal))

    #return np.array([mav, wl, zc, rms])
    # return np.array([mav, rms, var, zc, wl, ssc, wamp, iemg])
    return np.array([rms, wl, wamp])



def train_classifier(data_file, label_file, window_size, fs):
    """
    Entraîne un classifieur EMG avec normalisation
    """

    # ---------- 1. Charger les données ----------
    data = pd.read_csv(data_file)
    labels = pd.read_csv(label_file, header=None).values.flatten()

    signal1 = data["voltage1 (V)"].values
    signal2 = data["voltage2 (V)"].values

    # ---------- 2. Filtrage ----------
    signal1 = filter_emg(signal1, fs)
    signal2 = filter_emg(signal2, fs)

    # ---------- 3. Fenêtrage + features ----------
    X = []
    y = []

    for i in range(len(labels)):
        start = i * window_size
        end = start + window_size

        window1 = signal1[start:end]
        window2 = signal2[start:end]

        if len(window1) != window_size:
            continue

        f1 = extract_features(window1)
        f2 = extract_features(window2)

        feature_vector = np.concatenate([f1, f2])

        X.append(feature_vector)
        y.append(labels[i])

    X = np.array(X)
    y = np.array(y)

    # ---------- 4. NORMALISATION ----------
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # ---------- 5. Entraîner ----------
    # kNN
    #model = KNeighborsClassifier(n_neighbors=3)
    # Naive Bayes
    #model = GaussianNB()
    #LDA
    model = LinearDiscriminantAnalysis()
    # QDA
    #model = QuadraticDiscriminantAnalysis()
    # MLP
    #model = MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000)
    # SVM
    #model = SVC(kernel='rbf', C=10, gamma='scale')


    # --- ---
    model.fit(X_scaled, y)

    #print("Entraînement terminé")
    #print("Nombre d'échantillons :", len(X))

    # 👉 on retourne AUSSI le scaler !
    return model, scaler

# Test

### Offline

In [253]:
import pandas as pd
import numpy as np

def test_classifier_offline(data_file, model, scaler, window_size, fs):
    """
    Test du classifieur (simulation temps réel avec fichier CSV)

    Returns:
    - predictions : liste des classes prédites
    """

    # ---------- 1. Charger les données ----------
    data = pd.read_csv(data_file)

    signal1 = data["voltage1 (V)"].values
    signal2 = data["voltage2 (V)"].values

    # ---------- 2. Filtrage ----------
    signal1 = filter_emg(signal1, fs)
    signal2 = filter_emg(signal2, fs)

    # ---------- 3. Fenêtrage + prédiction ----------
    predictions = []

    num_samples = len(signal1)
    num_windows = num_samples // window_size

    for i in range(num_windows):
        start = i * window_size
        end = start + window_size

        window1 = signal1[start:end]
        window2 = signal2[start:end]

        # features
        f1 = extract_features(window1)
        f2 = extract_features(window2)

        feature_vector = np.concatenate([f1, f2])

        # ---------- NORMALISATION ----------
        feature_vector = scaler.transform([feature_vector])

        # ---------- PRÉDICTION ----------
        pred = model.predict(feature_vector)[0]

        predictions.append(pred)

    return predictions

### Online

In [221]:
import numpy as np
import time

def test_classifier_online(ch1, ch2, model, scaler, window_size, fs, buzzer):
    """
    Classification EMG en temps réel

    Parameters:
    - ch1, ch2 : canaux ADC (AnalogIn)
    - model : classifieur entraîné
    - scaler : scaler entraîné
    - window_size : taille de fenêtre
    - fs : fréquence d'échantillonnage
    - buzzer : instance PWM
    """

    buffer1 = []
    buffer2 = []

    step_size = window_size // 2  # overlap 50%

    print("Démarrage classification temps réel...")

    while True:
        # ---------- 1. Acquisition ----------
        sample1 = ch1.voltage
        sample2 = ch2.voltage

        buffer1.append(sample1)
        buffer2.append(sample2)

        # ---------- 2. Si fenêtre pleine ----------
        if len(buffer1) >= window_size:

            # prendre la fenêtre
            window1 = np.array(buffer1[:window_size])
            window2 = np.array(buffer2[:window_size])

            # ---------- 3. Filtrage ----------
            window1 = filter_emg(window1, fs)
            window2 = filter_emg(window2, fs)

            # ---------- 4. Features ----------
            f1 = extract_features(window1)
            f2 = extract_features(window2)

            feature_vector = np.concatenate([f1, f2])

            # ---------- 5. Normalisation ----------
            feature_vector = scaler.transform([feature_vector])

            # ---------- 6. Prédiction ----------
            pred = model.predict(feature_vector)[0]

            # ---------- 7. Feedback buzzer ----------
            if pred == 1:  # flexion
                buzzer.ChangeFrequency(1000)
                buzzer.start(50)

            elif pred == 2:  # extension
                buzzer.ChangeFrequency(2000)
                buzzer.start(50)

            else:  # pas de mouvement
                buzzer.stop()

            print("Prediction :", pred)

            # ---------- 8. Fenêtre glissante ----------
            buffer1 = buffer1[step_size:]
            buffer2 = buffer2[step_size:]

        # ---------- 9. Petite pause ----------
        time.sleep(1 / fs)

# Run

## Evaluation

In [254]:
import pandas as pd
import numpy as np

def evaluate_model(test_data_file, test_label_file, model, scaler, window_size, fs):
    """
    Évalue le modèle avec la balanced accuracy

    Returns:
    - score : balanced accuracy
    - predictions : liste des prédictions
    """

    # ---------- 1. Prédictions ----------
    predictions = test_classifier_offline(test_data_file, model, scaler, window_size, fs)
    predictions = np.array(predictions)

    # ---------- 2. Charger les labels ----------
    true_labels = pd.read_csv(test_label_file, header=None).values.flatten()

    # ⚠️ alignement taille
    true_labels = true_labels[:len(predictions)]

    # ---------- 3. Balanced accuracy ----------
    classes = np.unique(true_labels)
    acc_per_class = []

    for c in classes:
        idx = (true_labels == c)

        if np.sum(idx) == 0:
            continue

        acc = np.mean(predictions[idx] == true_labels[idx])
        acc_per_class.append(acc)

    score = np.mean(acc_per_class)

    # ---------- 4. Affichage ----------
    #print("Balanced accuracy :", round(score, 3))

    #for i, c in enumerate(classes):
    #    print(f"Accuracy classe {c} :", round(acc_per_class[i], 3))

    return score, predictions

## Evaluation sur une configuration

In [255]:
train_data = "train1_data.csv"
train_labels = "train1_labels.csv"
test_data = "test1_data.csv"
test_labels = "test1_labels.csv"
window_size = 200
fs = 1000
#fs = acquire_training_dataset(...)

model, scaler = train_classifier(train_data, train_labels, window_size, fs)

score, predictions = evaluate_model(test_data, test_labels, model, scaler, window_size, fs)

print(score)

0.5395833333333333


## Evalutation - tableau

In [256]:
import pandas as pd

window_size = 50
fs = 1000 # Attention : fs >= 1000

num_train = 6
num_test = 6

results = []

# ---------- BOUCLE ----------
for i in range(1, num_train + 1):

    train_data = f"train{i}_data.csv"
    train_labels = f"train{i}_labels.csv"


    model, scaler = train_classifier(train_data, train_labels, window_size, fs)

    for j in range(1, num_test + 1):

        test_data = f"test{j}_data.csv"
        test_labels = f"test{j}_labels.csv"

        score, _ = evaluate_model(
            test_data, test_labels, model, scaler, window_size, fs
        )

        results.append({
            "train": f"train{i}",
            "test": f"test{j}",
            "score": score
        })

# ---------- TABLEAU ----------
df = pd.DataFrame(results)

pivot_table = df.pivot(index="train", columns="test", values="score")

print("\n=== Balanced Accuracy Table ===")
print(pivot_table)

# ---------- MOYENNE GLOBALE ----------
mean_score = df["score"].mean()

print("\n=== Moyenne globale (balanced accuracy) ===")
print(round(mean_score, 3))


=== Balanced Accuracy Table ===
test       test1     test2     test3     test4     test5     test6
train                                                             
train1  0.817787  0.756335  0.396905  0.535444  0.707072  0.771717
train2  0.728059  0.851584  0.569286  0.563746  0.680859  0.504545
train3  0.484043  0.500000  0.877659  0.741247  0.489796  0.570960
train4  0.468085  0.496154  0.671052  0.683030  0.474490  0.469231
train5  0.799867  0.680769  0.472222  0.496101  0.733052  0.648776
train6  0.764881  0.514706  0.468750  0.556530  0.627551  0.851457

=== Moyenne globale (balanced accuracy) ===
0.623


In [225]:
import itertools
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import balanced_accuracy_score

# =========================================================
# Available feature names
# =========================================================
ALL_FEATURES = [
    "mav",
    "rms",
    "var",
    "zc",
    "wl",
    "ssc",
    "wamp",
    "iemg",
]


def extract_features(signal, selected_features=None, wamp_threshold=0.01):
    """
    Extract EMG features for one window.

    Parameters
    ----------
    signal : np.ndarray
        1D signal window
    selected_features : list[str] or tuple[str], optional
        Features to keep. If None, uses all features.
    wamp_threshold : float
        Threshold for WAMP

    Returns
    -------
    np.ndarray
        Feature vector in the same order as selected_features
    """
    if selected_features is None:
        selected_features = ALL_FEATURES

    # Compute all features once
    feature_dict = {
        "mav": np.mean(np.abs(signal)),
        "zc": np.sum(signal[:-1] * signal[1:] < 0),
        "wl": np.sum(np.abs(np.diff(signal))),
        "rms": np.sqrt(np.mean(signal ** 2)),
        "ssc": np.sum(((signal[1:-1] - signal[:-2]) * (signal[1:-1] - signal[2:])) > 0),
        "var": np.var(signal),
        "wamp": np.sum(np.abs(signal[1:] - signal[:-1]) > wamp_threshold),
        "iemg": np.sum(np.abs(signal)),
    }

    return np.array([feature_dict[name] for name in selected_features], dtype=float)


def build_feature_matrix(data_file, window_size, fs, selected_features=None):
    """
    Build X from a data CSV file.
    """
    data = pd.read_csv(data_file)

    signal1 = data["voltage1 (V)"].values
    signal2 = data["voltage2 (V)"].values

    # Filtering
    signal1 = filter_emg(signal1, fs)
    signal2 = filter_emg(signal2, fs)

    X = []
    num_samples = len(signal1)
    num_windows = num_samples // window_size

    for i in range(num_windows):
        start = i * window_size
        end = start + window_size

        window1 = signal1[start:end]
        window2 = signal2[start:end]

        if len(window1) != window_size or len(window2) != window_size:
            continue

        f1 = extract_features(window1, selected_features=selected_features)
        f2 = extract_features(window2, selected_features=selected_features)

        feature_vector = np.concatenate([f1, f2])
        X.append(feature_vector)

    return np.array(X)


def train_classifier(data_file, label_file, window_size, fs, selected_features=None):
    """
    Train EMG classifier with configurable features.
    """
    # ---------- 1. Load labels ----------
    labels = pd.read_csv(label_file, header=None).values.flatten()

    # ---------- 2. Build features ----------
    X = build_feature_matrix(
        data_file=data_file,
        window_size=window_size,
        fs=fs,
        selected_features=selected_features,
    )

    # Make labels match the number of valid windows
    y = labels[:len(X)]

    # ---------- 3. Normalize ----------
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # ---------- 4. Train ----------
    model = LinearDiscriminantAnalysis()
    model.fit(X_scaled, y)

    return model, scaler


def test_classifier_offline(data_file, model, scaler, window_size, fs, selected_features=None):
    """
    Offline test with configurable features.

    Returns
    -------
    list
        Predicted labels
    """
    X = build_feature_matrix(
        data_file=data_file,
        window_size=window_size,
        fs=fs,
        selected_features=selected_features,
    )

    X_scaled = scaler.transform(X)
    predictions = model.predict(X_scaled)

    return predictions.tolist()


def evaluate_model(test_data_file, test_label_file, model, scaler, window_size, fs, selected_features=None):
    """
    Evaluate one trained model on one test file.
    """
    y_true = pd.read_csv(test_label_file, header=None).values.flatten()

    predictions = test_classifier_offline(
        test_data_file,
        model,
        scaler,
        window_size,
        fs,
        selected_features=selected_features,
    )

    y_true = y_true[:len(predictions)]
    score = balanced_accuracy_score(y_true, predictions)

    return score, predictions


def generate_feature_combinations(feature_names, min_features=1, max_features=None):
    """
    Generate all feature combinations.
    """
    if max_features is None:
        max_features = len(feature_names)

    combinations = []
    for r in range(min_features, max_features + 1):
        combinations.extend(itertools.combinations(feature_names, r))

    return combinations


def evaluate_all_feature_combinations(
    num_train,
    num_test,
    window_size,
    fs,
    min_features=1,
    max_features=None,
):
    """
    Try all feature combinations and return the best one.

    Returns
    -------
    results_df : pd.DataFrame
        Mean score for every feature combination
    best_features : tuple
        Best feature combination
    best_score : float
        Best mean balanced accuracy
    """
    feature_combinations = generate_feature_combinations(
        ALL_FEATURES,
        min_features=min_features,
        max_features=max_features,
    )

    all_results = []

    for selected_features in feature_combinations:
        print(f"Testing features: {selected_features}")

        results = []

        for i in range(1, num_train + 1):
            train_data = f"train{i}_data.csv"
            train_labels = f"train{i}_labels.csv"

            model, scaler = train_classifier(
                train_data,
                train_labels,
                window_size,
                fs,
                selected_features=selected_features,
            )

            for j in range(1, num_test + 1):
                test_data = f"test{j}_data.csv"
                test_labels = f"test{j}_labels.csv"

                score, _ = evaluate_model(
                    test_data,
                    test_labels,
                    model,
                    scaler,
                    window_size,
                    fs,
                    selected_features=selected_features,
                )

                results.append(score)

        mean_score = float(np.mean(results))

        all_results.append({
            "features": selected_features,
            "num_features": len(selected_features),
            "mean_score": mean_score,
        })

        print(f" -> mean balanced accuracy = {mean_score:.4f}\n")

    results_df = pd.DataFrame(all_results)
    results_df = results_df.sort_values(by="mean_score", ascending=False).reset_index(drop=True)

    best_features = results_df.loc[0, "features"]
    best_score = results_df.loc[0, "mean_score"]

    return results_df, best_features, best_score

In [226]:
results_df, best_features, best_score = evaluate_all_feature_combinations(
    num_train=num_train,
    num_test=num_test,
    window_size=window_size,
    fs=fs,
    min_features=1,
    max_features=8,   # or smaller if you want fewer combinations
)

print("\n=== Top feature combinations ===")
print(results_df.head(20))

print("\n=== Best combination ===")
print("Features:", best_features)
print("Balanced accuracy:", round(best_score, 4))

Testing features: ('mav',)
 -> mean balanced accuracy = 0.5973

Testing features: ('rms',)
 -> mean balanced accuracy = 0.5998

Testing features: ('var',)
 -> mean balanced accuracy = 0.5975

Testing features: ('zc',)
 -> mean balanced accuracy = 0.4496

Testing features: ('wl',)
 -> mean balanced accuracy = 0.3752

Testing features: ('ssc',)
 -> mean balanced accuracy = 0.2682

Testing features: ('wamp',)
 -> mean balanced accuracy = 0.3351

Testing features: ('iemg',)
 -> mean balanced accuracy = 0.5973

Testing features: ('mav', 'rms')
 -> mean balanced accuracy = 0.5895

Testing features: ('mav', 'var')
 -> mean balanced accuracy = 0.6005

Testing features: ('mav', 'zc')
 -> mean balanced accuracy = 0.6024

Testing features: ('mav', 'wl')
 -> mean balanced accuracy = 0.6168

Testing features: ('mav', 'ssc')
 -> mean balanced accuracy = 0.5841

Testing features: ('mav', 'wamp')
 -> mean balanced accuracy = 0.5642

Testing features: ('mav', 'iemg')
 -> mean balanced accuracy = 0.5973